In [1]:
import googleapiclient.discovery
import pandas as pd
import os
from dotenv import load_dotenv
from data_classes import RawComment
from datetime import datetime

load_dotenv()
dev = os.getenv("BYD_KEY")

api_service_name = "youtube"
api_version = "v3"
DEVELOPER_KEY = dev


youtube = googleapiclient.discovery.build(
    api_service_name, api_version, developerKey=DEVELOPER_KEY)


In [ ]:
# video_id = "w_tCOgxXKwA"
# request = youtube.commentThreads().list(
#   part="snippet",
#   videoId = video_id,
#   maxResults = 100
# )

# raw_comments = []
# response = request.execute()
# print(response)

{'kind': 'youtube#commentThreadListResponse', 'etag': 'hBfA6JNtp-vIQcmSbbPxsFo_Iy0', 'nextPageToken': 'Z2V0X25ld2VzdF9maXJzdC0tQ2dnSWdBUVZGN2ZST0JJRkNLZ2dHQUFTQlFpSklCZ0FFZ1VJaUNBWUFCSUZDSjBnR0FFU0JRaUhJQmdBSWc0S0RBaTUxY3JPQmhDZ19lZnBBUQ==', 'pageInfo': {'totalResults': 100, 'resultsPerPage': 100}, 'items': [{'kind': 'youtube#commentThread', 'etag': 'nOIcZ3XALDq5hX_xquTl2Vvz-l0', 'id': 'UgxiHcDD-ATXV6d-Mtl4AaABAg', 'snippet': {'channelId': 'UC29JbxEwr7q5bP7ANJMSqAg', 'videoId': 'w_tCOgxXKwA', 'topLevelComment': {'kind': 'youtube#comment', 'etag': 'p-8Nvq6XBrLtgOB822cxMdK16Mg', 'id': 'UgxiHcDD-ATXV6d-Mtl4AaABAg', 'snippet': {'channelId': 'UC29JbxEwr7q5bP7ANJMSqAg', 'videoId': 'w_tCOgxXKwA', 'textDisplay': 'I still thinks they must have more buttons and less menues.', 'textOriginal': 'I still thinks they must have more buttons and less menues.', 'authorDisplayName': '@charruaporelmundo', 'authorProfileImageUrl': 'https://yt3.ggpht.com/ytc/AIdro_lnuI14TIaxPJ-oU5EqrHBJL1F2tcHs7Daib2qVyoR3I

In [ ]:
def parse_top_comments(video_id):
  request = youtube.commentThreads().list(
    part='snippet',
    videoId= video_id,
    maxResults = 100
  )
  raw_comments = []
  response = request.execute()

  for comment in response["items"]:
    raw_comment = RawComment(
      comment_id = comment['id'],
      video_id= comment["snippet"]["videoId"],
      parent_id= None,
      raw_text= comment["snippet"]["topLevelComment"]["snippet"]["textOriginal"],
      language= None, #need to change
      likes= comment["snippet"]["topLevelComment"]["snippet"]["likeCount"],
      created_at= datetime.fromisoformat(comment["snippet"]["topLevelComment"]["snippet"]["publishedAt"].replace("Z","+00:00"))
    )
    raw_comments.append(raw_comment)
  return raw_comments
# print(raw_comments[0])

In [ ]:
def parse_replies(reply_response, video_id):
    reply_request = youtube.comments().list(
        part="snippet",
        parentId="Ugxy1BklwU-MV6FEXRR4AaABAg",
        maxResults=100
        )
    reply_response = reply_request.execute()

    raw_replies = []
    for reply in reply_response["items"]:
        raw_reply = RawComment(
            comment_id = reply['id'],
            video_id= video_id,
            parent_id= reply["snippet"]["parentId"],
            raw_text= reply["snippet"]["textOriginal"],
            language= None, #need to change
            likes= reply["snippet"]["likeCount"],
            created_at= datetime.fromisoformat(reply["snippet"]["publishedAt"].replace("Z","+00:00"))
        )
        raw_replies.append(raw_reply)
    return raw_replies


In [ ]:
def extract_comments